In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib
import sklearn
print(os.getcwd())

BUILDING_NAME = 'building1.csv'

ASCENDING_SPEED = 1.0
DESCENDING_SPEED = 2.0
HORIZONTAL_SPEED = 1.5

ASCENDING_ENERGY= 50
DESCENDING_ENERGY= 5
HORIZONTAL_ENERGY = 10

BATTERY_ONE = 3600  # 1 Wh = 3600 J
BATTERY_TWO = 21600 # 6 Wh = 21600 J

/home/michele/big-project-FOR


# Reading CSV and creating pairs


In [ ]:
#loading building csv data
building = pd.read_csv(BUILDING_NAME)

#generating all possible pairs of points (edges)
nodes_pairs = building.rename_axis('source_node').reset_index().merge(building.rename_axis('target_node').reset_index(), how='cross')

#removing self-self pairs
nodes_pairs = nodes_pairs[nodes_pairs['source_node'] != nodes_pairs['target_node']] #keeping only pairs with different points

In [76]:
first_point_cols= ['x_x','y_x','z_x']
second_point_cols=['x_y','y_y','z_y']
diff_cols= ['dx','dy','dz']

# Computing distances 

In [77]:
#function to determine if two points are connected based on distance and coordinate deltas

def process_connections(dataframe):
    #distances and deltas on coordinates
    dataframe[diff_cols] = (dataframe[second_point_cols].values - dataframe[first_point_cols].values)
    dataframe['euclidean_distance'] = np.sqrt((dataframe[diff_cols]**2).sum(axis=1)) #axis=1 sum on columns 
    
    
    # conditions
    condition_near = (dataframe['euclidean_distance'] <= 4.0)
    
    
    condition_tunnel = (
        ((dataframe['dx'].abs() <= 0.5) & (dataframe['dy'].abs() <= 0.5)) | 
        ((dataframe['dy'].abs() <= 0.5) & (dataframe['dz'].abs() <= 0.5)) | 
        ((dataframe['dx'].abs() <= 0.5) & (dataframe['dz'].abs() <= 0.5))
    )
    condition_tunnel_and_not_far = (dataframe['euclidean_distance'] <= 11.0) & condition_tunnel
    
    # final connection condition
    dataframe["are_connected"] = condition_near | condition_tunnel_and_not_far
    return dataframe



In [ ]:
nodes_pairs = process_connections(nodes_pairs)

nodes_pairs.head()

,source_node,x_x,y_x,z_x,target_node,x_y,y_y,z_y,dx,dy,dz,euclidean_distance,are_connected
1,0,2.5,3.25,0.0,1,-6.0,3.25,0.0,-8.5,0.0,0.0,8.500000,True
2,0,2.5,3.25,0.0,2,-6.0,11.75,0.0,-8.5,8.5,0.0,12.020815,False
3,0,2.5,3.25,0.0,3,2.5,11.75,0.0,0.0,8.5,0.0,8.500000,True
4,0,2.5,3.25,0.0,4,2.5,3.25,11.0,0.0,0.0,11.0,11.000000,True
5,0,2.5,3.25,0.0,5,-6.0,3.25,11.0,-8.5,0.0,11.0,13.901439,False


In [ ]:
#computing vertical distances on z and distances on horizontal plane x,y
nodes_pairs["d_horizontal"] = np.sqrt(nodes_pairs['dx']**2 + nodes_pairs['dy']**2)
nodes_pairs["d_vertical"] = nodes_pairs['dz']

#dropping columns that are not useful anymore
nodes_pairs.drop(columns=['dx','dy','dz'], inplace=True)
nodes_pairs.drop(columns=['x_x','y_x','z_x','x_y','y_y','z_y'], inplace=True)

#keeping only connected pairs
nodes_pairs = nodes_pairs[nodes_pairs['are_connected']==True]
nodes_pairs.drop(columns=['are_connected'], inplace=True)

nodes_pairs.head(10)

,source_node,target_node,euclidean_distance,d_horizontal,d_vertical
1,0,1,8.500000,8.500000,0.0
3,0,3,8.500000,8.500000,0.0
4,0,4,11.000000,0.000000,11.0
18,0,18,4.007805,4.007805,0.0
19,0,19,1.030776,1.030776,0.0
64,1,0,8.500000,8.500000,0.0
66,1,2,8.500000,8.500000,0.0
69,1,5,11.000000,0.000000,11.0
82,1,18,4.506939,4.506939,0.0
83,1,19,7.504166,7.504166,0.0


# Function for time travel

In [ ]:
#finding rows corresponding to different movement types 

d_horizontal = nodes_pairs['d_horizontal']
d_vertical = nodes_pairs['d_vertical']
rows_horizontal = (d_vertical == 0)
rows_ascending = (d_horizontal == 0) & (d_vertical > 0)
rows_descending = (d_horizontal == 0) & (d_vertical < 0)
rows_oblique_ascending = (d_horizontal != 0) & (d_vertical > 0)
rows_oblique_descending = (d_horizontal != 0) & (d_vertical < 0)

In [81]:
#function for time travel based on movement types and speeds

def compute_time_travel(dataframe):
    dataframe.loc[rows_horizontal, 'time_travel'] = dataframe.loc[rows_horizontal, 'd_horizontal'] / HORIZONTAL_SPEED

    dataframe.loc[rows_ascending, 'time_travel'] = dataframe.loc[rows_ascending, 'd_vertical'] / ASCENDING_SPEED
    dataframe.loc[rows_descending, 'time_travel'] = -dataframe.loc[rows_descending, 'd_vertical'] / DESCENDING_SPEED

    dataframe.loc[rows_oblique_ascending, 'time_travel'] = np.maximum(dataframe.loc[rows_oblique_ascending, 'd_horizontal'] / HORIZONTAL_SPEED,
                                                                     dataframe.loc[rows_oblique_ascending, 'd_vertical'] / ASCENDING_SPEED)
    
    dataframe.loc[rows_oblique_descending, 'time_travel'] = np.maximum(dataframe.loc[rows_oblique_descending, 'd_horizontal'] / HORIZONTAL_SPEED,
                                                                     -dataframe.loc[rows_oblique_descending, 'd_vertical'] / DESCENDING_SPEED)
    return dataframe

# Function for energy consumption


In [82]:
#function for energy cost based on movement types and energy consumption rates

def compute_energy_cost(dataframe):
    dataframe.loc[rows_horizontal, 'energy_consumption'] = dataframe.loc[rows_horizontal, 'd_horizontal'] * HORIZONTAL_ENERGY

    dataframe.loc[rows_ascending, 'energy_consumption'] = dataframe.loc[rows_ascending, 'd_vertical'] * ASCENDING_ENERGY
    dataframe.loc[rows_descending, 'energy_consumption'] = -dataframe.loc[rows_descending, 'd_vertical'] * DESCENDING_ENERGY

    dataframe.loc[rows_oblique_ascending, 'energy_consumption'] = (dataframe.loc[rows_oblique_ascending, 'd_horizontal'] * HORIZONTAL_ENERGY +
                                                                  dataframe.loc[rows_oblique_ascending, 'd_vertical'] * ASCENDING_ENERGY)
    
    dataframe.loc[rows_oblique_descending, 'energy_consumption'] = (dataframe.loc[rows_oblique_descending, 'd_horizontal'] * HORIZONTAL_ENERGY +
                                                                  -dataframe.loc[rows_oblique_descending, 'd_vertical'] * DESCENDING_ENERGY)
    return dataframe

# Actually computing energy and time costs for edges

In [ ]:
#computing time travel and energy cost for each pair (edges)

nodes_pairs = compute_time_travel(nodes_pairs)

nodes_pairs = compute_energy_cost(nodes_pairs)

nodes_pairs.drop(columns=['d_horizontal','d_vertical', 'euclidean_distance'], inplace=True)

nodes_pairs.head(10)


,source_node,target_node,time_travel,energy_consumption
1,0,1,5.666667,85.000000
3,0,3,5.666667,85.000000
4,0,4,11.000000,550.000000
18,0,18,2.671870,40.078049
19,0,19,0.687184,10.307764
64,1,0,5.666667,85.000000
66,1,2,5.666667,85.000000
69,1,5,11.000000,550.000000
82,1,18,3.004626,45.069391
83,1,19,5.002777,75.041655
